# Module 16: Writing Up a Causal Claim

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Fifteen modules produced one number and a dozen checks. This module turns
them into the thing that leaves the building.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

keep = [a for a in TRAINED if a != "A007"]
prof = profile.set_index("agency_id")

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated_ids, outcome="n_uof", offset=None):
    """The standard specification, with whoever is labelled treated."""
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated_ids))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated_ids))
                  & (s["period"] == "phase")).astype(float)
    off = s["lo"] if offset is None else offset
    z = smf.glm(f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase", s,
                family=sm.families.Poisson(), offset=off).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z.bse["settled"]

## 2. Every estimate this series produced

In [ ]:
dall = f.copy()
dall["lo"] = np.log(dall["n_arrests"])
tb = cell_rate(TRAINED, "before")
ta = cell_rate(TRAINED, "after")

s = d.copy()
s["settled"] = ((s["agency_id"].isin(keep)) & (s["period"] == "after")).astype(float)
s["phase"] = ((s["agency_id"].isin(keep)) & (s["period"] == "phase")).astype(float)
z_notime = smf.glm("n_uof ~ C(agency_id) + settled + phase", s,
                   family=sm.families.Poisson(), offset=s["lo"]).fit()

rows = [
    {"analysis": "before and after, the five trained agencies",
     "estimate": f"{100 * (ta / tb - 1):+.1f}%", "why it is wrong": "no comparison group"},
    {"analysis": "agency effects, nothing absorbing time",
     "estimate": f"{pct(z_notime.params['settled']):+.1f}%",
     "why it is wrong": "the settled term picks up the statewide decline"},
    {"analysis": "difference in differences, all five trained",
     "estimate": f"{fit(dall, TRAINED)[0]:+.1f}%",
     "why it is wrong": "one agency was on its own pre trend"},
    {"analysis": "difference in differences, checked",
     "estimate": f"{fit(d, keep)[0]:+.1f}%", "why it is wrong": ""},
    {"analysis": "THE TRUTH", "estimate": f"{TRUTH:+.1f}%", "why it is wrong": ""},
]
pd.DataFrame(rows).set_index("analysis")

Every one was computed correctly. They differ only in which sources of
variation the analyst accounted for.

## 3. The checks, assembled

In [ ]:
e, lo, hi, se = fit(d, keep)
pre = d[d["period"] == "before"]
d2 = pre.copy()
d2["tr"] = d2["agency_id"].isin(keep).astype(float)
zp = smf.glm("n_uof ~ C(agency_id) + yr + tr:yr", d2,
             family=sm.families.Poisson(), offset=d2["lo"]).fit()
kk = [x for x in zp.params.index if "yr" in x and "tr" in x][0]
plo, phi = [pct(v) for v in zp.conf_int().loc[kk]]

s2 = d.copy()
s2["settled"] = ((s2["agency_id"].isin(keep)) & (s2["period"] == "after")).astype(float)
s2["phase"] = ((s2["agency_id"].isin(keep)) & (s2["period"] == "phase")).astype(float)
za = smf.glm("n_arrests ~ C(agency_id)+C(year_month)+settled+phase", s2,
             family=sm.families.Poisson()).fit()
alo, ahi = [pct(v) for v in za.conf_int().loc["settled"]]

checks = [
    ("estimate", f"{e:+.1f}%  [{lo:+.1f}, {hi:+.1f}]"),
    ("smallest detectable effect", f"{100 * (1 - np.exp(-2.80 * se)):.1f}%"),
    ("pre program trend difference",
     f"{pct(zp.params[kk]):+.2f}% a year  [{plo:+.2f}, {phi:+.2f}]"),
    ("placebo on the denominator, arrests",
     f"{pct(za.params['settled']):+.2f}%  [{alo:+.2f}, {ahi:+.2f}]"),
    ("agencies excluded", "1 treated, for a pre trend of -12.0% a year"),
    ("months excluded", "1, documented civil unrest at Tarnbridge"),
    ("hidden trend needed to zero the estimate", "about -3.4% a year"),
]
for lab, val in checks:
    print(f"  {lab:44s} {val}")

## 4. The paragraph

In [ ]:
print(f"""
Across {d['agency_id'].nunique()} agencies and {d['year_month'].nunique()} months, use of force at the four
agencies that adopted de escalation training ran {abs(e):.1f} percent below the
comparison agencies over the {int(s2['settled'].sum() / 4)} months after the training was fully in
place, 95 percent interval from {abs(hi):.1f} to {abs(lo):.1f} percent below. Rates are
incidents per arrest.

One treated agency was excluded because its use of force rate was already
falling at 12.0 percent a year before the program began, against 4 to 6
percent elsewhere; including it raises the estimate to 17.0 percent. One
month of documented civil unrest was excluded. The pre program trends of the
two groups differ by {abs(pct(zp.params[kk])):.2f} percent a year, interval from {abs(plo):.2f} below to
{abs(phi):.2f} above.

The design could have detected a reduction of {100 * (1 - np.exp(-2.80 * se)):.1f} percent or larger.
Arrests, the denominator, were unaffected ({pct(za.params['settled']):+.2f} percent), so the result
is not an artefact of changing arrest practice. A randomisation test over 400
reassignments of the treatment label places the estimate beyond 97 percent of
what the procedure produces from chance alone.

The estimate is robust to contamination of the comparison group but not to an
unmeasured trend difference: a hidden trend of 3.4 percent a year favouring
the treated agencies would eliminate it, and the pre period cannot exclude a
difference that large. The agencies were selected for the program on the
basis of their pre program use of force rate, which is this study's outcome.
""")

Around 230 words. Every clause is doing work, and a reader can locate the
weak points without being told where to look.

**What it does not contain**: the word "caused", any claim about agencies
outside the study, and any decimal place the interval does not support.

## 5. The checklist

**Before estimating**
- [ ] The comparison group rule is written down
- [ ] The intervention date is fixed
- [ ] The smallest detectable effect is computed and reported to whoever commissioned the work

**Before believing the estimate**
- [ ] Pre program trends, agency by agency, with intervals
- [ ] The estimate under every defensible comparison group rule
- [ ] A leave one out check if one agency dominates
- [ ] Placebo dates, placebo outcomes, placebo groups
- [ ] Whether the program moved the denominator

**Before writing**
- [ ] Every exclusion listed with its reason and its cost
- [ ] The interval, not just the estimate
- [ ] How recipients were selected
- [ ] A sensitivity result for the main assumption
- [ ] Nothing causal that the design cannot support

## Exercise

Write the one sentence version, the kind that goes in a press release, and
check it against the checklist.

In [ ]:
# Fill in the blank, then run.
SHOW = None          # try True

if SHOW:
    print(f"""
    One sentence, defensible:

      "Use of force fell {abs(e):.0f} percent more at the four agencies that adopted the
       training than at comparable agencies over the same period, with a range
       of {abs(hi):.0f} to {abs(lo):.0f} percent."

    One sentence, not defensible, and the difference is two words:

      "The training cut use of force by {abs(e):.0f} percent."
    """)
else:
    print("Set SHOW above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
SHOW = True
```

The two words are **cut** and the absence of a comparison. The first sentence
says the rate fell more at one set of agencies than another; the second says
the training was the reason.

The first also carries the range, which is what stops a reader treating 12.6
as a measurement. Everything else in the report supports the first sentence
and none of it establishes the second.

**A press release will shorten whatever you give it.** Write the shortest
defensible version yourself, and it is the one that gets used.

---

## Where this goes next

This level built one estimate and checked it. The
[Advanced series](../../Advanced/) takes up what these checks leave open: what
identification means formally, why the designs that were not available here
are not available, what happens when adoption is staggered, how to do
inference with a handful of clusters, and how far partial identification can
take an answer when the assumptions cannot be defended.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*